# PDF Extraction Demo

## Flow

PDF file → Load PDF → Extract text page by page → Count pages → Count characters → Detect empty pages → Save extracted text → Generate metadata → Generate ingestion log

## 1. Import Required Libraries

In [1]:
import json
import uuid
import shutil
from pathlib import Path
from datetime import datetime, timezone

try:
    import fitz  # PyMuPDF
    PDF_ENGINE = "pymupdf"
except ImportError:
    fitz = None
    PDF_ENGINE = None

## 2. Define Project Paths

In [2]:
def find_project_root(current_path: Path) -> Path:
    """
    Find project root by walking upward until the data/ folder is found.
    This makes the notebook work even if it is executed from notebooks/data_team/.
    """
    current_path = current_path.resolve()

    for path in [current_path] + list(current_path.parents):
        if (path / "data").exists():
            return path

    raise FileNotFoundError(
        "Could not find project root. Please make sure a 'data/' folder exists in the project."
    )


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

sample_input_dir = PROJECT_ROOT / "data" / "sample_inputs"

# Preferred simple filename for Week 2 demo.
preferred_input_path = sample_input_dir / "sample.pdf"

# Fallback to the uploaded resume PDF filename if you use the original name.
fallback_input_path = sample_input_dir / "big-data-engineer2 - Template 16 .pdf"

if preferred_input_path.exists():
    input_path = preferred_input_path
elif fallback_input_path.exists():
    input_path = fallback_input_path
else:
    pdf_files = list(sample_input_dir.glob("*.pdf"))
    input_path = pdf_files[0] if pdf_files else preferred_input_path

raw_dir = PROJECT_ROOT / "data" / "raw" / "pdf"
staging_dir = PROJECT_ROOT / "data" / "staging" / "pdf"
log_dir = PROJECT_ROOT / "logs"

raw_dir.mkdir(parents=True, exist_ok=True)
staging_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

raw_output_path = raw_dir / "sample_pdf_raw.pdf"
staging_text_output_path = staging_dir / "sample_pdf_text.txt"
log_output_path = log_dir / "pdf_ingestion_log.json"
metadata_output_path = log_dir / "pdf_metadata.json"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Input path:", input_path)
print("Input exists:", input_path.exists())
print("Raw output:", raw_output_path)
print("Staging text output:", staging_text_output_path)
print("Ingestion log output:", log_output_path)
print("Metadata output:", metadata_output_path)

Current dir: f:\data\new\quanskill\DataVision_Duy\week2\notebooks\data_team
Project root: F:\data\new\quanskill\DataVision_Duy\week2
Input path: F:\data\new\quanskill\DataVision_Duy\week2\data\sample_inputs\big-data-engineer2 - Template 16 .pdf
Input exists: True
Raw output: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\pdf\sample_pdf_raw.pdf
Staging text output: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\pdf\sample_pdf_text.txt
Ingestion log output: F:\data\new\quanskill\DataVision_Duy\week2\logs\pdf_ingestion_log.json
Metadata output: F:\data\new\quanskill\DataVision_Duy\week2\logs\pdf_metadata.json


## 3. Validate Input PDF File

In [3]:
if not input_path.exists():
    raise FileNotFoundError(
        f"Input PDF file not found. Expected a PDF file in: {sample_input_dir}"
    )

if input_path.stat().st_size == 0:
    raise ValueError(f"Input PDF file is empty: {input_path}")

if input_path.suffix.lower() != ".pdf":
    raise ValueError(f"Input file is not a PDF: {input_path}")

if PDF_ENGINE is None:
    raise ImportError(
        "PyMuPDF is not installed. Please install it with: pip install pymupdf"
    )

print("Input PDF validation passed.")
print("PDF extraction engine:", PDF_ENGINE)

Input PDF validation passed.
PDF extraction engine: pymupdf


## 4. Start Ingestion Run

In [4]:
run_id = str(uuid.uuid4())
source_name = "sample_pdf"
source_type = "pdf"
owner = "Nguyen Minh Duy"

start_time = datetime.now(timezone.utc).isoformat()

print("Run ID:", run_id)
print("Start time:", start_time)

Run ID: b30ab525-f696-4de2-af14-fea28415332b
Start time: 2026-06-01T05:00:57.585937+00:00


## 5. Load Sample PDF

In [5]:
try:
    pdf_document = fitz.open(input_path)
    status = "success"
    error_message = None

    page_count = pdf_document.page_count

    print("PDF loaded successfully.")
    print("Page count:", page_count)

except Exception as error:
    status = "failed"
    error_message = str(error)
    raise

PDF loaded successfully.
Page count: 1


## 6. Save Raw PDF Copy

In [6]:
# Raw layer should preserve the original PDF file.
shutil.copy2(input_path, raw_output_path)

print("Raw PDF copied to:", raw_output_path)

Raw PDF copied to: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\pdf\sample_pdf_raw.pdf


## 7. Extract Text Page by Page

In [7]:
page_texts = []
empty_pages = []

for page_index in range(page_count):
    page = pdf_document.load_page(page_index)
    text = page.get_text("text")

    page_number = page_index + 1
    character_count = len(text)

    if text.strip() == "":
        empty_pages.append(page_number)

    page_texts.append({
        "page_number": page_number,
        "text": text,
        "character_count": character_count,
        "is_empty": text.strip() == ""
    })

print("Text extraction completed.")
print("Pages extracted:", len(page_texts))
print("Empty pages:", empty_pages)

Text extraction completed.
Pages extracted: 1
Empty pages: []


## 8. Count Extracted Characters

In [8]:
total_extracted_characters = sum(page["character_count"] for page in page_texts)
records_read = page_count
records_valid = page_count - len(empty_pages)
records_invalid = len(empty_pages)

print("Total pages:", page_count)
print("Total extracted characters:", total_extracted_characters)
print("Valid pages:", records_valid)
print("Empty/invalid pages:", records_invalid)

Total pages: 1
Total extracted characters: 2650
Valid pages: 1
Empty/invalid pages: 0


## 9. Preview Extracted Text

In [9]:
preview_text = page_texts[0]["text"][:1000] if page_texts else ""

print(preview_text)

First Last
Big Data Engineer
Wichita, Kansas • +1-234-456-789 • professionalemail@resumeworded.com • linkedin.com/in/username
WORK EXPERIENCE
_______________________________________________________________________________________________________________
Resume Worded, New York, NY
09/2015 – Present
Augmented reality startup with 50+ employees and $100m+ annual revenue
Big Data Engineer
●
Designed scalable big data processing pipelines using Hadoop, Spark, and other distributed computing
technologies to process 100+ Petabytes monthly.
●
Created ETL (extract, transform, load) workflows to extract data from 30+ sources, transform it into a
standardized format, and load it into a data warehouse for further analysis.
●
Built and maintained 70+ dashboards and reports to monitor data quality, usage, and performance metrics
using Grafana, Kibana, and Tableau.
●
Partnered with 110+ data scientists to translate their needs into technical requirements, data models, and
data architecture.
Polyhire

## 10. Save Extracted Text to Staging

In [10]:
with open(staging_text_output_path, "w", encoding="utf-8") as file:
    for page in page_texts:
        file.write(f"===== Page {page['page_number']} =====\n")
        file.write(page["text"])
        file.write("\n\n")

print("Extracted text saved to:", staging_text_output_path)

Extracted text saved to: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\pdf\sample_pdf_text.txt


## 11. Generate Document Metadata

In [11]:
document_metadata = {
    "run_id": run_id,
    "source_name": source_name,
    "source_type": source_type,
    "file_name": input_path.name,
    "input_path": str(input_path),
    "file_size_bytes": int(input_path.stat().st_size),
    "page_count": int(page_count),
    "total_extracted_characters": int(total_extracted_characters),
    "empty_pages": empty_pages,
    "empty_page_count": int(len(empty_pages)),
    "extraction_engine": PDF_ENGINE,
    "raw_output_path": str(raw_output_path),
    "staging_text_output_path": str(staging_text_output_path),
    "owner": owner
}

with open(metadata_output_path, "w", encoding="utf-8") as file:
    json.dump(document_metadata, file, indent=4, ensure_ascii=False)

print("PDF metadata saved to:", metadata_output_path)
document_metadata

PDF metadata saved to: F:\data\new\quanskill\DataVision_Duy\week2\logs\pdf_metadata.json


{'run_id': 'b30ab525-f696-4de2-af14-fea28415332b',
 'source_name': 'sample_pdf',
 'source_type': 'pdf',
 'file_name': 'big-data-engineer2 - Template 16 .pdf',
 'input_path': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\sample_inputs\\big-data-engineer2 - Template 16 .pdf',
 'file_size_bytes': 63047,
 'page_count': 1,
 'total_extracted_characters': 2650,
 'empty_pages': [],
 'empty_page_count': 0,
 'extraction_engine': 'pymupdf',
 'raw_output_path': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\raw\\pdf\\sample_pdf_raw.pdf',
 'staging_text_output_path': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\staging\\pdf\\sample_pdf_text.txt',
 'owner': 'Nguyen Minh Duy'}

## 12. Generate Ingestion Log

In [12]:
end_time = datetime.now(timezone.utc).isoformat()

ingestion_log = {
    "run_id": run_id,
    "source_name": source_name,
    "source_type": source_type,
    "input_path_or_url": str(input_path),
    "start_time": start_time,
    "end_time": end_time,
    "status": status,
    "records_read": int(records_read),
    "records_valid": int(records_valid),
    "records_invalid": int(records_invalid),
    "error_message": error_message,
    "raw_output_path": str(raw_output_path),
    "staging_output_path": str(staging_text_output_path),
    "clean_output_path": None,
    "owner": owner,
    "page_count": int(page_count),
    "total_extracted_characters": int(total_extracted_characters),
    "empty_pages": empty_pages,
    "metadata_output_path": str(metadata_output_path)
}

with open(log_output_path, "w", encoding="utf-8") as file:
    json.dump(ingestion_log, file, indent=4, ensure_ascii=False)

print("PDF ingestion log saved to:", log_output_path)
ingestion_log

PDF ingestion log saved to: F:\data\new\quanskill\DataVision_Duy\week2\logs\pdf_ingestion_log.json


{'run_id': 'b30ab525-f696-4de2-af14-fea28415332b',
 'source_name': 'sample_pdf',
 'source_type': 'pdf',
 'input_path_or_url': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\sample_inputs\\big-data-engineer2 - Template 16 .pdf',
 'start_time': '2026-06-01T05:00:57.585937+00:00',
 'end_time': '2026-06-01T05:01:20.150740+00:00',
 'status': 'success',
 'records_read': 1,
 'records_valid': 1,
 'records_invalid': 0,
 'error_message': None,
 'raw_output_path': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\raw\\pdf\\sample_pdf_raw.pdf',
 'staging_output_path': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\staging\\pdf\\sample_pdf_text.txt',
 'clean_output_path': None,
 'owner': 'Nguyen Minh Duy',
 'page_count': 1,
 'total_extracted_characters': 2650,
 'empty_pages': [],
 'metadata_output_path': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\logs\\pdf_metadata.json'}

## 13. Close PDF Document

In [13]:
pdf_document.close()
print("PDF document closed.")

PDF document closed.


## 14. Final Output Check

In [14]:
print("Expected outputs:")

print("Raw output exists:", raw_output_path.exists())
print("Staging text output exists:", staging_text_output_path.exists())
print("Ingestion log exists:", log_output_path.exists())
print("Metadata output exists:", metadata_output_path.exists())

print("\nOutput paths:")
print("Raw:", raw_output_path)
print("Staging text:", staging_text_output_path)
print("Ingestion log:", log_output_path)
print("Metadata:", metadata_output_path)

Expected outputs:
Raw output exists: True
Staging text output exists: True
Ingestion log exists: True
Metadata output exists: True

Output paths:
Raw: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\pdf\sample_pdf_raw.pdf
Staging text: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\pdf\sample_pdf_text.txt
Ingestion log: F:\data\new\quanskill\DataVision_Duy\week2\logs\pdf_ingestion_log.json
Metadata: F:\data\new\quanskill\DataVision_Duy\week2\logs\pdf_metadata.json


## 15. Summary

```text
data/raw/pdf/sample_pdf_raw.pdf
data/staging/pdf/sample_pdf_text.txt
logs/pdf_ingestion_log.json
logs/pdf_metadata.json
```

PDF extraction only creates staging text output. A clean PDF text layer can be added later after text normalization, chunking, or document cleaning.